In [1]:
import torch
print(f"cuda available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")

cuda available=True
gpu=NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
from pathlib import Path
import sys
from torch import nn
start=Path.cwd().resolve()
project=next(p for p in (start,*start.parents) if (p/"src"/"model.py").is_file())
if str(project) not in sys.path:
    sys.path.insert(0,str(project))
from src.model import PolicyNetwork
torch.manual_seed(42)
assert torch.cuda.is_available(),"cuda unavailable"
model=PolicyNetwork(hidden_layers=((16,nn.Tanh),),hidden_bias=False,output_bias=False,input_size=4,output_size=2).to("cuda")
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
print(f"model device={next(model.parameters()).device}")
print(f"parameters={sum(p.numel() for p in model.parameters())}")

model device=cuda:0
parameters=96


In [3]:
import gymnasium as gym
from time import perf_counter
from src.train import train
env=gym.make("CartPole-v1")
env.reset(seed=42)
started=perf_counter()
try:
    history=train(model,env,optimizer,episodes=1000,batch_size=5,keep_episodes=False,log_every=20)
finally:
    env.close()
print(f"training seconds={perf_counter()-started:.1f}")

episodes=100 | mean score=23.0
episodes=200 | mean score=20.0
episodes=300 | mean score=17.8
episodes=400 | mean score=37.6
episodes=500 | mean score=25.0
episodes=600 | mean score=36.2
episodes=700 | mean score=35.4
episodes=800 | mean score=44.8
episodes=900 | mean score=91.8
episodes=1000 | mean score=154.8
training seconds=37.8


In [5]:
from src.evaluate import evaluate
eval_env=gym.make("CartPole-v1")
try:
    results=evaluate(model,eval_env,episodes=100,deterministic=True,seed=10000)
finally:
    eval_env.close()
print(f"mean score={results['mean_score']:.1f}")
print(f"std score={results['std_score']:.1f}")
print(f"min score={results['min_score']:.1f}")
print(f"max score={results['max_score']:.1f}")

mean score=393.8
std score=97.6
min score=212.0
max score=500.0


In [6]:
checkpoint_dir=project/"checkpoints"
checkpoint_dir.mkdir(exist_ok=True)
torch.save({
    "model_state_dict":model.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    "model_config":{"hidden_layers":[(16,"Tanh")],"hidden_bias":False,"output_bias":False,"input_size":4,"output_size":2},
    "episodes_completed":1000,
    "history":history,
    "evaluation":results
},checkpoint_dir/"experiment_001_1000episodes.pt")
results_before=results.copy()

In [7]:
env=gym.make("CartPole-v1")
env.reset(seed=43)
try:
    history_more=train(model,env,optimizer,episodes=1000,batch_size=5,keep_episodes=False,log_every=20)
finally:
    env.close()

episodes=100 | mean score=185.6
episodes=200 | mean score=170.0
episodes=300 | mean score=313.8
episodes=400 | mean score=327.8
episodes=500 | mean score=346.6
episodes=600 | mean score=335.0
episodes=700 | mean score=367.2
episodes=800 | mean score=428.4
episodes=900 | mean score=443.8
episodes=1000 | mean score=496.2


In [8]:
eval_env=gym.make("CartPole-v1")
try:
    results=evaluate(model,eval_env,episodes=100,deterministic=True,seed=10000)
finally:
    eval_env.close()
for metric in ("mean_score","std_score","min_score","max_score"):
    print(f"{metric}: before={results_before[metric]:.1f} | after={results[metric]:.1f}")

mean_score: before=393.8 | after=500.0
std_score: before=97.6 | after=0.0
min_score: before=212.0 | after=500.0
max_score: before=500.0 | after=500.0


In [9]:
checkpoint_dir=project/"checkpoints"
checkpoint_dir.mkdir(exist_ok=True)
torch.save({
    "model_state_dict":model.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    "model_config":{"hidden_layers":[(16,"Tanh")],"hidden_bias":False,"output_bias":False,"input_size":4,"output_size":2},
    "episodes_completed":2000,
    "history_first":history,
    "history_more":history_more,
    "evaluation":results
},checkpoint_dir/"experiment_001_2000episodes.pt")
print("checkpoint saved")

checkpoint saved


In [10]:
eval_env=gym.make("CartPole-v1")
try:
    new_results=evaluate(model,eval_env,episodes=100,deterministic=True,seed=20000)
finally:
    eval_env.close()
for metric in ("mean_score","std_score","min_score","max_score"):
    print(f"{metric}={new_results[metric]:.1f}")

mean_score=500.0
std_score=0.0
min_score=500.0
max_score=500.0
